# Cell 1 — Research configuration and environment

**Occlusion- and Uncertainty-Aware Real-Time Human Detection and Tracking in Thermal
Imagery from Unmanned Aerial Vehicles for Search and Rescue**

This PhD project studies anonymous human detection in aerial thermal imagery for civilian
search and rescue. Small apparent targets, occlusion, and domain shifts motivate controlled
experiments. The title states the research objective; real-time performance and improved
accuracy remain unproven. Identification, weapon detection, hostile-person classification,
and targeting are outside scope. Thermal-only inference is the primary setting.

**Purpose:** establish the research protocol, explicit baseline, configuration, and
reproducibility records before reading annotations or training.

**Inputs/dependencies:** this notebook, `src/config.py`, `src/reproducibility.py`, and
`configs/base.yaml`; Python 3.11+ with PyYAML and nbformat. Torch, torchvision, NumPy,
Pillow, and pycocotools are inspected for future readiness. No dataset is required here.

**Outputs:** a unique experiment directory containing effective configuration and its
SHA-256, environment and baseline-source records, package inventory, seed settings,
dependency readiness, source-file hashes, and a run manifest. No detector is built and
no performance results are produced. Runtime category: **environment smoke test**.

Run all cells in order from a fresh kernel. Shared code and configuration must be available
alongside the notebook, locally or as an attached Kaggle code dataset. Cell 6 discovers
those files; explicit hints resolve ambiguous mount names.

## Cell 2 — Research questions and testable hypotheses

1. Does a high-resolution pathway improve recall for small persons relative to the unmodified
   detector under the same splits, training budget, input policy, and pretraining?
2. Does image-dependent adaptive multi-scale fusion outperform fixed fusion and globally
   learned static weights? Compare AP and subgroup recall across completed seeds.
3. Does masked visibility supervision improve detection of genuinely occluded persons where
   valid annotations exist? Report annotation counts and agreement before training this head.
4. Do localization distributions and held-out confidence calibration improve appropriate
   likelihood/coverage diagnostics and calibration under the frozen matching protocol?
5. What accuracy, memory, and end-to-end latency trade-offs are observed on measured hardware?

Each is a falsifiable hypothesis, including the possibility of no benefit or degraded results.
Metadata, temporal processing, and identity metrics require verified supporting data.

## Cell 3 — Proposed contributions and established baseline

The primary hypothesis is **input-dependent adaptive multi-scale feature fusion with
high-resolution features and auxiliary visibility supervision**. Reliability gating,
probabilistic localization, held-out calibration, metadata, temporal fusion, tracking,
and deployment are later investigations. All switches are disabled in this notebook.

**Concrete reference:** `torchvision.models.detection.fcos_resnet50_fpn`, the established
FCOS detector with ResNet-50 and FPN. Its source has separable `backbone.body`,
`backbone.fpn`, and `head` interfaces. The inspected torchvision 0.22.1 implementation
returns ResNet layers `[2, 3, 4]` (C3–C5), produces 256-channel P3–P5, and adds
P6–P7 with `LastLevelP6P7(256, 256)`: nominal strides 8, 16, 32, 64, 128.
It does not include P2. These source-derived properties motivate testing a finer pathway;
they do not establish that it will help. Cell 7 records the implementation installed at run time.

FCOS is anchor-free in formulation; torchvision uses one anchor-like reference point per
location for assignment and box decoding. Its head includes classification, regression,
and centerness, so the eventual reference loss includes centerness as well.
Inspect actual tensors and run forward/backward tests in Notebook 06 before modifying it.

**Thermal input proposal:** repeat a decoded single thermal channel three times to retain
the reference encoder. This uses no RGB measurements. Confirm bit depth, intensity scaling,
and normalization in Notebooks 02/05; thermal values are not assumed to be temperatures.
Provisional resize settings retain the factory's 800-pixel short-side / 1333-pixel long-side
policy, not a fixed square. Freeze that choice after validation, before final-test inspection.

COCO_V1 weights are accessible upstream and proposed for the later baseline. Here the actual
weight source is `none`; downloads are disabled. A configured local file can be validated
without loading it. Notebook 06 must document state-dict compatibility, person-class mapping,
head adaptation, preprocessing, and all pretraining differences. Torchvision code is BSD-3-Clause;
dataset and pretrained-weight terms require their own review. The detection API is marked beta.

Primary sources:
- [FCOS source for torchvision v0.22.1](https://github.com/pytorch/vision/blob/v0.22.1/torchvision/models/detection/fcos.py)
- [Backbone/FPN source](https://github.com/pytorch/vision/blob/v0.22.1/torchvision/models/detection/backbone_utils.py)
- [Factory and weights documentation](https://docs.pytorch.org/vision/stable/generated/torchvision.models.detection.fcos_resnet50_fpn.html)
- [Torchvision code license](https://github.com/pytorch/vision/blob/main/LICENSE)
- [Reference detection evaluator](https://github.com/pytorch/vision/tree/v0.22.1/references/detection)
- [HIT-UAV source and annotation description](https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset)
- [HIT-UAV license](https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/blob/main/LICENSE)

HIT-UAV is the initial candidate dataset. Its upstream description includes standard and
oriented boxes and collection-condition information; this does not prove that a mounted
copy contains verified per-image metadata. Verify the actual version, labels, official
splits, and license in Notebook 02. MONET/WiSARD external evaluation and BIRDSAI/video
tracking remain conditional on source and local-data inspection. No dataset is downloaded.

## Cell 4 — Evaluation protocol and metrics

**Primary:** person AP averaged over IoU 0.50:0.05:0.95 using one consistent COCO-style
evaluator. Also report AP50 and person recall at a development-validation-selected score
threshold. Report false positives per evaluated image (FP/image), including empty scenes;
do not confuse this denominator with FP/(FP+TN), whose detection TN population is undefined.

Report target-size groups in **original-image coordinates**; freeze category cutoffs during
data analysis. Visibility/altitude/day–night groups require verified labels and sample counts.
Preserve official benchmark splits, and report additional research splits separately.
Audit duplicates and candidate near-duplicates; use flight/sequence/location disjointness
only with verified groups. Keep training, development validation, held-out calibration,
internal test, and external test distinct. No final-test labels guide model selection,
preprocessing statistics, calibration, thresholds, or hyperparameters.

**Secondary, when implemented:** calibration error and reliability curves under explicit
detection matching/population rules; localization NLL and coverage; HOTA/IDF1/switches/
fragmentation only with ground-truth identities. Anonymous tracker IDs are not person identity.
Repeat principal comparisons across seeds where feasible, report actual runs, and resample
verified groups rather than correlated frames as independent observations.

Measure batch-one latency with warm-up, GPU synchronization when applicable, median/tail
latency, memory, and separate preprocessing/model/postprocessing/tracking times. CPU, Kaggle
GPU, and actual embedded-device results remain distinct. Model FPS alone does not prove
end-to-end real-time operation. No metrics, confidence intervals, or energy measurements exist yet.

## Cell 5 — Stages, dependencies, and artifact contracts

| Stage | Notebooks | Entry requirement |
|---|---|---|
| Configuration | **01 (this notebook only)** | Shared source and base configuration |
| Audit, visibility, splits, loaders | 02–05 | Mounted data and verified provenance |
| A: unmodified baseline | 06 | Frozen splits, tested transforms/loaders |
| B–D: high resolution, adaptive fusion, visibility | 07–10 | Working baseline and valid supervision |
| E: probabilistic localization and calibration | 08–12 | Tested loss and separate calibration data |
| F: optional metadata | 07/10 | Verified metadata plus tested missing-data fallback |
| G: temporal processing/tracking | 13 | Causal sequences; identities for identity metrics |
| Failure analysis, deployment, statistics, packaging | 14–17 | Frozen protocol and real completed runs |

Run Notebook 01 independently from a fresh kernel. Later notebooks must load saved contracts,
never inherit variables. Approval is required before beginning Notebook 02. No training occurs here.

Draft manifest contracts are recorded as version identifiers only in `config.contracts`:
image rows will contain dataset/image IDs, image path, dimensions, split, verified grouping
IDs, optional camera/altitude fields, and availability indicators; object rows will contain
dataset/image IDs, optional source object IDs, original/mapped class, xyxy coordinates,
optional visibility and availability flag, and track IDs only when ground truth exists.
Missing values remain null/masked. Row validators and real manifests belong to Notebook 02.
The split identifier/fingerprint remain null until real split contents are frozen in Notebook 04.
A configuration hash is not a dataset checksum or split fingerprint.

In [ ]:
import os
from pathlib import Path

project_root = Path("/kaggle/input/datasets/dule11/thermal-project-root")
required = ["src/config.py", "src/reproducibility.py", "configs/base.yaml"]

missing = [name for name in required if not (project_root / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files: {missing}")

os.environ["THERMAL_PROJECT_ROOT"] = str(project_root)
print("Verified project root:", project_root)

Verified project root: /kaggle/input/datasets/dule11/thermal-project-root


In [ ]:
# Cell 6 — Core imports and project discovery. Edit hints if automatic discovery is ambiguous.
import copy
import hashlib
import json
import logging
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

PROJECT_ROOT_HINT = os.environ.get("THERMAL_PROJECT_ROOT")  # actual code mount/root, if needed
DATASET_ROOT_HINT = os.environ.get("THERMAL_DATASET_ROOT")  # actual inspected dataset directory
CONFIG_PATH_HINT = os.environ.get("THERMAL_CONFIG_PATH")  # optional writable YAML copy

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
# Set before any torch import/CUDA initialization; do not pretend to reseed interpreter hashes.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

if PROJECT_ROOT_HINT:
    candidates = [Path(PROJECT_ROOT_HINT).expanduser().resolve()]
else:
    candidates = [p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "configs" / "base.yaml").is_file()]
    if not candidates and Path("/kaggle/input").is_dir():
        candidates = sorted({p.parent.parent for pattern in
                             ("*/configs/base.yaml", "*/*/configs/base.yaml", "*/*/*/configs/base.yaml")
                             for p in Path("/kaggle/input").glob(pattern)})
candidates = [p for p in candidates
              if (p / "src" / "config.py").is_file() and (p / "configs" / "base.yaml").is_file()]
if len(candidates) != 1:
    raise RuntimeError("Set PROJECT_ROOT_HINT to the actual code root containing src/ and configs/. "
                       f"Candidates: {[str(p) for p in candidates]}")
PROJECT_ROOT = candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import (
    config_hash,
    create_run_directory,
    load_config,
    resolve_paths,
    save_json,
    validate_config,
)
from src.reproducibility import dependency_readiness, inspect_environment, seed_everything

CONFIG_PATH = Path(CONFIG_PATH_HINT).expanduser().resolve() if CONFIG_PATH_HINT else PROJECT_ROOT / "configs/base.yaml"
print("Project root:", PROJECT_ROOT)
print("Configuration:", CONFIG_PATH)

Project root: /kaggle/input/datasets/dule11/thermal-project-root
Configuration: /kaggle/input/datasets/dule11/thermal-project-root/configs/base.yaml


In [ ]:
# Cell 7 — Inspect the current interpreter, binary compatibility, hardware, and baseline source.
environment = inspect_environment()
print(json.dumps({key: environment[key] for key in
                  ("python", "platform", "cpu", "logical_cpu_count", "packages", "cuda", "errors")}, indent=2))
source_record = environment["baseline_source"]
print("FCOS source inspection:", {key: value for key, value in source_record.items()
                                 if key != "factory_source"})
print("CPU NMS check:", environment["torchvision_nms_cpu"])
print("No detector construction, weight loading, or GPU model test was performed.")

{
  "python": {
    "version": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
    "executable": "/usr/bin/python3"
  },
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "cpu": "x86_64",
  "logical_cpu_count": 4,
  "packages": {
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "numpy": "2.1.3",
    "PyYAML": "6.0.3",
    "Pillow": "13.0.0",
    "nbformat": "5.11.1",
    "nbclient": "0.5.13",
    "ipykernel": "6.17.1",
    "pycocotools": "2.0.11",
    "pytest": "8.4.2",
    "ruff": null,
    "mypy": null,
    "bandit": null
  },
  "cuda": {
    "available": true,
    "build_version": "13.0",
    "cudnn_version": 92700,
    "devices": [
      {
        "index": 0,
        "name": "Tesla T4",
        "memory_bytes": 15636037632,
        "compute_capability": [
          7,
          5
        ]
      },
      {
        "index": 1,
        "name": "Tesla T4",
        "memory_bytes": 15636037632,
        "compute_capability": [
          7,
          5
      

In [ ]:
# Cell 8 — Load seed controls early; Cell 9 exposes the same central configuration.
bootstrap_config = load_config(CONFIG_PATH)
seed_report = seed_everything(
    bootstrap_config["seed"],
    deterministic=bootstrap_config["hardware"]["deterministic_algorithms"],
    benchmark=bootstrap_config["hardware"]["cudnn_benchmark"],
)
print(json.dumps(seed_report, indent=2))

{
  "seed": 42,
  "python_seeded": true,
  "pythonhashseed_at_process_start": null,
  "hash_seed_note": "Setting PYTHONHASHSEED now cannot reseed this interpreter's hashes.",
  "cublas_workspace_config": ":4096:8",
  "numpy_error": null,
  "numpy_seeded": true,
  "torch_error": null,
  "torch_seeded": true,
  "cuda_seeded": true,
  "deterministic_algorithms": true,
  "cudnn_benchmark": false
}


In [ ]:
# Cell 9 — Central experiment configuration. Edit base.yaml (or a writable copy) before Run All.
# CONFIG_PATH_HINT can select that copy on read-only Kaggle code mounts.
config = copy.deepcopy(bootstrap_config)
validate_config(config)
print(json.dumps(config, indent=2))
print("Training, calibration, proposed architecture, metadata, and tracking are disabled.")

{
  "schema_version": "1.0",
  "experiment_name": "thermal-sar-foundation",
  "seed": 42,
  "dataset": {
    "dataset_id": "HIT-UAV",
    "root": null,
    "version": null,
    "license_verified_locally": false
  },
  "split": {
    "identifier": null,
    "fingerprint": null
  },
  "model": {
    "variant": "torchvision_fcos_resnet50_fpn",
    "thermal_input": "repeat_single_channel_three_times",
    "input_min_size": 800,
    "input_max_size": 1333,
    "weights_source": "none",
    "local_weights_path": null,
    "proposed_pretrained_weights": "FCOS_ResNet50_FPN_Weights.COCO_V1",
    "allow_weight_download": false
  },
  "training": {
    "enabled": false,
    "mode": "smoke",
    "batch_size": 2,
    "epochs": 1,
    "learning_rate": 0.005,
    "optimizer": "SGD",
    "momentum": 0.9,
    "weight_decay": 0.0001,
    "scheduler": "StepLR",
    "scheduler_step_size": 1,
    "scheduler_gamma": 0.1,
    "gradient_clip_norm": 10.0,
    "early_stopping_patience": 5,
    "num_workers": 0


In [ ]:
# Cell 10 — Resolve actual mounts and writable output paths without guessing a dataset folder.
if DATASET_ROOT_HINT:
    config["dataset"]["root"] = str(Path(DATASET_ROOT_HINT).expanduser().resolve())
paths = resolve_paths(config, PROJECT_ROOT)
config["dataset"]["root"] = paths["dataset_root"]
for field in ("input_root", "working_root", "output_root"):
    config["paths"][field] = paths[field]
validate_config(config)
print(json.dumps(paths, indent=2))
print("Mount names are candidates only; dataset identity and annotation contents are not audited here.")

{
  "environment": "kaggle",
  "project_root": "/kaggle/input/datasets/dule11/thermal-project-root",
  "input_root": "/kaggle/input",
  "working_root": "/kaggle/working",
  "output_root": "/kaggle/working/artifacts/experiments",
  "dataset_root": null,
  "mounts": [
    "datasets"
  ],
  "dataset_status": "not_selected"
}
Mount names are candidates only; dataset identity and annotation contents are not audited here.


In [ ]:
# Cell 11 — Content identity is separate from each execution's unique experiment ID.
digest = config_hash(config)
run_metadata = {
    "schema_version": "1.0",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "notebook": "01_Research_Configuration_and_Environment.ipynb",
    "configuration_sha256": digest,
    "split_fingerprint": config["split"]["fingerprint"],
    "dataset_version": config["dataset"]["version"],
    "execution_environment": paths["environment"],
    "training_started": False,
}
print("Effective configuration SHA-256:", digest)
print("Split fingerprint:", run_metadata["split_fingerprint"], "(not created yet)")

Effective configuration SHA-256: 6f4570ed14d3e457a7b30b7234b8dd754c6f3c0d60fd2a1136c98ba6e972dde3
Split fingerprint: None (not created yet)


In [ ]:
# Cell 12 — Reserve a new directory on every execution. Never resume into an old directory.
run_dir = create_run_directory(Path(paths["output_root"]), config["experiment_name"], digest)
run_metadata["experiment_id"] = run_dir.name
run_metadata["output_directory"] = str(run_dir)
checkpoint_dir = run_dir / "checkpoints"
print("Experiment ID:", run_dir.name)
print("Artifacts:", run_dir)
print("Checkpoints (empty; no training):", checkpoint_dir)

Experiment ID: thermal-sar-foundation_20261120T021724762181Z_6f4570ed14_753fd7de
Artifacts: /kaggle/working/artifacts/experiments/thermal-sar-foundation_20261120T021724762181Z_6f4570ed14_753fd7de
Checkpoints (empty; no training): /kaggle/working/artifacts/experiments/thermal-sar-foundation_20261120T021724762181Z_6f4570ed14_753fd7de/checkpoints


In [ ]:
# Cell 13 — Diagnose current and future dependencies; no installation is executed.
readiness = dependency_readiness(environment)
readiness["dataset_selected"] = paths["dataset_root"] is not None
readiness["dataset_audited"] = False
readiness["split_frozen"] = False
readiness["ready_for_training"] = False
print(json.dumps(readiness, indent=2))
print("Missing Notebook 01 packages: review `python -m pip install -r requirements.txt` before installing.")
if "pycocotools" in readiness["missing_detection_dependencies"]:
    print("Future evaluator dependency: proposed minimal command `python -m pip install pycocotools`.")
print("Select torch/torchvision together for the target platform; do not upgrade the Kaggle environment blindly.")

{
  "notebook01_ready": true,
  "missing_notebook01": [],
  "missing_detection_dependencies": [],
  "runtime_errors": [],
  "detection_environment_ready": true,
  "training_authorized": false,
  "installation_performed": false,
  "dataset_selected": false,
  "dataset_audited": false,
  "split_frozen": false,
  "ready_for_training": false
}
Missing Notebook 01 packages: review `python -m pip install -r requirements.txt` before installing.
Select torch/torchvision together for the target platform; do not upgrade the Kaggle environment blindly.


In [ ]:
# Cell 14 — Persist an immutable audit bundle. Re-running this cell refuses to overwrite files.
source_paths = [PROJECT_ROOT / "src/config.py", PROJECT_ROOT / "src/reproducibility.py", CONFIG_PATH]
notebook_path = PROJECT_ROOT / "notebooks/01_Research_Configuration_and_Environment.ipynb"
if notebook_path.is_file():
    source_paths.append(notebook_path)
source_hashes = {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in source_paths}
artifact_payloads = {
    "config.json": config,
    "environment.json": environment,
    "reproducibility.json": seed_report,
    "dependency_readiness.json": readiness,
    "source_hashes.json": source_hashes,
    "run.json": run_metadata,
}
targets = [run_dir / name for name in artifact_payloads] + [run_dir / "packages.txt"]
if any(path.exists() for path in targets):
    raise FileExistsError("Audit files already exist. Run all cells to create a fresh experiment directory.")
for name, payload in artifact_payloads.items():
    save_json(run_dir / name, payload)
with (run_dir / "packages.txt").open("x", encoding="utf-8") as handle:
    handle.write("# Inspected project dependencies only; not a complete environment lockfile.\n")
    for name, version in sorted(environment["packages"].items()):
        handle.write(f"{name}=={version}\n" if version else f"# {name}: not installed\n")
saved = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
assert config_hash(saved) == digest, "Saved configuration hash differs"
print("Saved and hash-verified:", ", ".join(path.name for path in targets))
print("Environment smoke test finished; dataset/training readiness is recorded separately.")

Saved and hash-verified: config.json, environment.json, reproducibility.json, dependency_readiness.json, source_hashes.json, run.json, packages.txt
Environment smoke test finished; dataset/training readiness is recorded separately.


## Cell 15 — Reproducibility limitations

Seeds and deterministic flags reduce variation; they do not guarantee bit-identical results
across PyTorch releases, CPU/GPU devices, kernels, or drivers. Strict deterministic algorithms
may reject unsupported operations later; investigate rather than silently disabling the setting.
`PYTHONHASHSEED` takes effect at interpreter launch, so setting it inside a running notebook
cannot reseed hash randomization. Set it before launching Jupyter when that control is required.
`CUBLAS_WORKSPACE_CONFIG` is set before imports here, but an already initialized CUDA context
in a reused kernel requires a kernel restart. Fresh NumPy `default_rng` objects need explicit
seeds; a legacy NumPy seed does not seed independently constructed generators. Data-loader
worker seeds, augmentation generators, and checkpoint RNG states belong to later notebooks.

The configuration SHA-256 covers effective configuration including resolved paths, not runtime
IDs or timestamps. Relocating data changes that hash. Run IDs are intentionally unique.
Source hashes and installed versions provide provenance; the small dependency inventory is
not a complete lockfile. Dataset version, grouping, split fingerprint, labels, and licenses
remain unverified until the corresponding audits. Missing visibility/metadata/identities
must remain unavailable; downstream experiments stay disabled until their requirements hold.

CPU NMS compatibility does not test a detector forward/backward pass, CUDA NMS, GPU training,
or embedded deployment. This notebook performs no training, downloads, installs, annotation
processing, threshold fitting, final-test evaluation, or model-performance measurement.

## Cell 16 — Readiness checklist and running instructions

- [ ] Run Cells 6–14 in order from a fresh kernel; inspect actual saved readiness/environment reports.
- [ ] Confirm selected paths and that the unique run folder is writable and outside input data.
- [ ] Confirm effective configuration hash, seed report, source record, and audit files exist.
- [ ] Resolve any import/binary errors before the corresponding future experiments.
- [ ] Before Notebook 02, attach the actual HIT-UAV copy and select its real root. Inspect version,
      class mapping, annotation format, official splits, local license, and available metadata there.
- [ ] Before training, complete manifest validators, leakage controls, frozen splits, loader/transform
      tests, shape/finite-loss/gradient tests, and one-batch overfitting. None are implemented here.

**Local:** open this notebook from the project root or `notebooks/` in Jupyter and select the
intended Python environment. Run All. Run shared tests from the project root with
`python -m unittest discover -s tests -v`.

**Kaggle:** upload/attach the project containing `src/`, `configs/`, and `notebooks/` as a code
dataset; import this notebook into a Python kernel. Cell 6 discovers actual code mounts under
`/kaggle/input`; if needed set `PROJECT_ROOT_HINT` there to the directory containing `src/`
and `configs/`. Attach HIT-UAV separately when available, and set `DATASET_ROOT_HINT` to its
actual root. For configuration edits, copy `configs/base.yaml` to `/kaggle/working/base.yaml`,
edit that copy, and set `CONFIG_PATH_HINT` in Cell 6. Do not modify `/kaggle/input`.
Internet and GPU are unnecessary for Notebook 01. Run All; generated artifacts go to
`/kaggle/working/artifacts/experiments/<unique-id>/`. Save/download that output after execution.

Runtime category: **smoke test**, normally seconds to a few minutes depending on imports
and storage. This is not a training-duration estimate. No installation commands execute.

Completion means this notebook's configuration/audit artifacts were written and verified,
not that the detector or data are ready for training. **Stop here.** Next is
`02_Dataset_Audit_and_Annotation_Standardization.ipynb`, only after explicit approval.